In [10]:
import duckdb
with duckdb.connect("../data/f1_market.duckdb") as con:
    con.sql("""
WITH t_before AS (
    SELECT 
        ticker,
        constructor,
        MIN(ticker_date) AS before_date,
        ticker_open as before_open,
        ticker_close as before_close,
        high as before_high,
        low as before_low,
        volume as before_volume
    FROM stg_ticker_data
    GROUP BY ticker, constructor, ticker_open, ticker_close, high, low, volume
),
t_after AS(
    SELECT 
        ticker,
        constructor,
        MAX(ticker_date) AS after_date,
        ticker_open as after_open,
        ticker_close as after_close,
        high as after_high,
        low as after_low,
        volume as after_volume
    FROM stg_ticker_data
    GROUP BY ticker, constructor, ticker_open, ticker_close, high, low, volume
)
SELECT 
    rr.constructor,
    rr.round,
    rr.race_name,
    rr.race_date,
    MIN(rr.position),
    b.ticker,
    b.before_close,
    a.after_close
FROM stg_race_results rr
INNER JOIN t_before b
ON rr.constructor = b.constructor
AND rr.race_date > b.before_date
INNER JOIN t_after a
ON rr.constructor = a.constructor
AND rr.race_date < a.after_date
WHERE rr.constructor LIKE '%mercedes%'
GROUP BY rr.constructor,rr.round, rr.race_name, rr.race_date, b.ticker, b.before_close, a.after_close
ORDER BY rr.round
            """).show()
    #con.sql("""
    #        SELECT * FROM raw_ticker_data
    #        """).show()

┌─────────────┬───────┬───────────────────────┬────────────┬────────────────────┬──────────┬────────────────────┬────────────────────┐
│ constructor │ round │       race_name       │ race_date  │ min(rr."position") │  ticker  │    before_close    │    after_close     │
│   varchar   │ int32 │        varchar        │    date    │       int32        │ varchar  │       double       │       double       │
├─────────────┼───────┼───────────────────────┼────────────┼────────────────────┼──────────┼────────────────────┼────────────────────┤
│ mercedes    │     1 │ Australian Grand Prix │ 2026-03-08 │                  1 │ LIGHT.AS │  17.11294174194336 │  593.8699951171875 │
│ mercedes    │     1 │ Australian Grand Prix │ 2026-03-08 │                  1 │ META     │  643.7117309570312 │               51.5 │
│ mercedes    │     1 │ Australian Grand Prix │ 2026-03-08 │                  1 │ LIGHT.AS │  17.11294174194336 │  400.6300048828125 │
│ mercedes    │     1 │ Australian Grand Prix │ 2026-03